In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../src/")
sys.path.append("/code/src/")

In [ ]:
from data_processing.core.raw_data_processing_pipeline import RawDataProcessingPipeline
from data_processing.core.camera_metadata_abc import GpsTagsMap

from data_processing.supported_capture_devices.dji_drone_mini_4 import (DjiDroneMini4ImageMetaData, 
                                                                        DjiDroneMini4TagsMap, 
                                                                        DjiDroneMini4VideoMetaData)

In [ ]:
from data_processing.core.raw_data_processing_pipeline import RawDataProcessingPipeline

## Main code

In [ ]:

scene_raw_dir = "/data/datasets/raw/backyard_sunny"
scene_description_file_name = "trajectories_metadata.json"
calibration_files_path = "/workspace/data/"

scene_processed_dir = "/data/datasets/processed/backyard_sunny_test2"
scene_processed_json_file_name = "scene_data.json"
scene_colmap_dir = "PYCOLMAP_soft_prior"

In [ ]:
min_distance_m = 0.3
min_rot_degree = 10
pos_covariance = [2, 2, 2]
max_num_models = 4
copy_images = True
run_colmap = False
copy_point_cloud = True
transform_world_coord = False
add_camera_center_distance_error = True
add_camera_center_components_error = True
add_camera_rotation_angle_error = True
add_camera_rotation_euler_error = True
add_statistics = True
use_fisheye_for_wfov = True
absolute_altitude = False

In [ ]:
# this is the configuration that need to be extended if more capture devices need to be supported
supported_image_capture_devices = {"dji_drone_mini_4_pro": lambda name, abs_alt: ( DjiDroneMini4ImageMetaData(image_file=name, 
                                                                                                        camera_tags_map=DjiDroneMini4TagsMap(),
                                                                                                        gps_tags_map=GpsTagsMap(),
                                                                                                        absolute_altitude=abs_alt)
                                                                                  )
                                    }

supported_video_capture_devices = {"dji_drone_mini_4_pro": lambda name, abs_alt: ( DjiDroneMini4VideoMetaData(video_file=name, 
                                                                                                        camera_tags_map=DjiDroneMini4TagsMap(),
                                                                                                        absolute_altitude=abs_alt)
                                                                                  )
                                  }


In [ ]:
# create and configure the pipeline
raw_data_pipeline = RawDataProcessingPipeline(supported_image_capture_devices=supported_image_capture_devices,
                                              supported_video_capture_devices=supported_video_capture_devices)

raw_data_pipeline.config_processing_pipeline(copy_images=copy_images, run_colmap=run_colmap, copy_point_cloud=copy_point_cloud,
                                             transform_world_coord=transform_world_coord, add_statistics=add_statistics, add_camera_center_distance_error=add_camera_center_distance_error,
                                             add_camera_center_components_error=add_camera_center_components_error, add_camera_rotation_angle_error=add_camera_rotation_angle_error,
                                             add_camera_rotation_euler_error=add_camera_rotation_euler_error, min_distance_m=min_distance_m, min_rot_degree=min_rot_degree, 
                                             pos_covariance=pos_covariance, max_num_models=max_num_models, wfov_as_fisheye=use_fisheye_for_wfov,
                                             absolute_altitude=absolute_altitude)

In [ ]:
# configure the scene info 
raw_data_pipeline.configure_scene(scene_raw_dir=scene_raw_dir, scene_processed_dir=scene_processed_dir,
                                  scene_description_file_name=scene_description_file_name, scene_processed_json_file_name=scene_processed_json_file_name,
                                  calibration_files_path=calibration_files_path, colmap_folder_name=scene_colmap_dir)

In [ ]:
# Run the raw data processing pipeline
raw_data_pipeline.process_scene_from_raw(debug_prints=True)

In [ ]:
# Print the trajectory statistics
raw_data_pipeline.print_trajectories_stats()